# Creating Download Scripts
We have the sample names of healthy and T2D donor beta cells.  We want to create a shell script to download only those files.

Luckily, the site (https://www.ebi.ac.uk/ena/browser/view/PRJEB15401?show=reads) gives us the download script for all 3514 samples.  We need only to filter it.  We are using the submitted read files.

The sample list was filtered using an excel file (manipulate_beta_cells.xlsx).

I have copy and pasted the sample names into 2 txt files.
1) healthy_beta.txt
2) t2d_beta.txt

## Read All Files into Memory

In [7]:
with open('healthy_beta.txt', 'r', encoding='utf-8') as f:
    healthy_beta = [line.strip() for line in f.readlines()]

print(f"{len(healthy_beta)} healthy beta cell samples")

171 healthy beta cell samples


In [8]:
with open('t2d_beta.txt', 'r', encoding='utf-8') as f:
    t2d_beta = [line.strip() for line in f.readlines()]

print(f"{len(t2d_beta)} T2D beta cell samples")

99 T2D beta cell samples


In [9]:
filepath = "shell_scripts/ena_manifest_PRJEB15401.sh"
with open(filepath, 'r', encoding='utf-8') as f:
    full_download_list = [line.strip() for line in f.readlines()]

full_download_list[:5]

['wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR163/ERR1632975/HP1525301T2D_J1.fastq.gz',
 'wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR163/ERR1630795/HP1504101T2D_N8.fastq.gz',
 'wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR163/ERR1631788/HP1507101_H19.fastq.gz',
 'wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR163/ERR1630319/HP1502401_I9.fastq.gz',
 'wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR163/ERR1633011/HP1525301T2D_K20.fastq.gz']

## Test if Filter Script Works

In [15]:
healthy_count, t2d_count = 0, 0

for line in full_download_list:
    _, filename = line.rsplit("/", 1)
    sample_name, _ = filename.split(".", 1)
    if sample_name in healthy_beta:
        healthy_count += 1
    if sample_name in t2d_beta:
        t2d_count += 1

print(f"{healthy_count} healthy beta cell samples")
print(f"{t2d_count} t2d beta cell samples")

171 healthy beta cell samples
99 t2d beta cell samples


The process of extracting the right download lines seems to work.  We got the same counts as above.  Let's now create the .sh files.

## Create .sh Files

In [17]:
with open("download_healthy_beta_samples.sh", "w", encoding="utf-8") as f:
    for line in full_download_list:
        _, filename = line.rsplit("/", 1)
        sample_name, _ = filename.split(".", 1)
        if sample_name in healthy_beta:
            f.write(line + "\n")


In [18]:
with open("download_t2d_beta_samples.sh", "w", encoding="utf-8") as f:
    for line in full_download_list:
        _, filename = line.rsplit("/", 1)
        sample_name, _ = filename.split(".", 1)
        if sample_name in t2d_beta:
            f.write(line + "\n")


## Copy to HPC and Run Scripts
Transfer the two `.sh` files to your HPC shared folder and run them to download the FASTQ files.

```bash
# Copy scripts to HPC
scp *.sh <username>@<hpc-hostname>:/path/to/files/
ssh <username>@<hpc-hostname>
cd /path/to/files/
chmod +x *.sh

./download_healthy_beta_samples.sh
ls -1 | wc -l   # expect 171

./download_t2d_beta_samples.sh
ls -1 | wc -l   # expect 99

gunzip *.fastq.gz
```

## Create a csv of Read Counts
Let's check the read counts for each file.  Perhaps we can run some statistics on them later.

Save the following as `record_readcounts.sh` on the HPC, then run it:

```bash
#!/bin/bash

# Output header
echo "directory,filename,read_count" > read_counts_filtered.csv

# Find all .fastq files recursively
find . -type f -name "*.fastq" | while read file; do
    read_count=$(wc -l < "$file")
    read_count=$((read_count / 4))
    directory=$(dirname "$file")
    filename=$(basename "$file")
    echo "$directory,$filename,$read_count" >> read_counts_filtered.csv
done

chmod +x record_readcounts.sh
./record_readcounts.sh
```

Then copy the result back locally:

```bash
scp <username>@<hpc-hostname>:/path/to/files/read_counts_filtered.csv ./
```

In [ ]:
import pandas as pd

df = pd.read_csv(r"read_counts_filtered.csv")

In [15]:
healthy_samples = (df["directory"] == "./healthy").sum()
t2d_samples = (df["directory"] == "./t2d").sum()

healthy_samples_enough_reads = ((df["directory"] == "./healthy") & (df["read_count"] >= 750_000)).sum()
t2d_samples_enough_reads = ((df["directory"] == "./t2d") & (df["read_count"] >= 750_000)).sum()

In [25]:
df_summary = pd.DataFrame({
    "Group": ["Healthy", "T2D"],
    "Total Samples": [healthy_samples, t2d_samples],
    "Samples with Enough Reads": [healthy_samples_enough_reads, t2d_samples_enough_reads],
})

df_summary["Percent with Enough Reads"] = (
    df_summary["Samples with Enough Reads"] / df_summary["Total Samples"] * 100
).round(0).astype(int).astype(str) + "%"

df_summary

,Group,Total Samples,Samples with Enough Reads,Percent with Enough Reads
0,Healthy,171,104,61%
1,T2D,99,45,45%


## Remove Files with Fewer Than 750,000 Reads
The paper removed all files with fewer than 750,000 reads.  Let us do this now.

Save the following as `cull_short_fastq.sh` on the HPC, then run it to remove samples below the read threshold:

```bash
#!/bin/bash

MIN_READS=750000

tail -n +2 read_counts_filtered.csv | while IFS=',' read -r directory filename read_count; do
    read_count=$(echo "$read_count" | xargs)
    filepath="$directory/$filename"
    if (( read_count < MIN_READS )); then
        echo "Removing: $filepath (reads: $read_count)"
        rm -f "$filepath"
    fi
done

chmod +x cull_short_fastq.sh
./cull_short_fastq.sh

ls -1 *.fastq | wc -l
```

104 healthy and 45 t2d files remain as expected.